In [97]:
from google.colab import files

uploaded = files.upload()

Saving test_fe.csv to test_fe (3).csv
Saving train_fe.csv to train_fe (3).csv


In [98]:
from google.colab import files

uploaded = files.upload()

Saving sample_submission.csv to sample_submission (3).csv


In [99]:
!pip install -q optuna xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.9 MB/s eta 0:00:00


In [100]:
import warnings
warnings.filterwarnings("ignore")

import json
import optuna
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier

In [101]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [102]:
train = pd.read_csv("train_fe.csv")
test = pd.read_csv("test_fe.csv")

print(train.shape)
print(test.shape)

(577347, 19)
(247435, 18)


In [103]:
for df in [train, test]:

    df["alpha_rad"] = np.radians(df["alpha"])
    df["delta_rad"] = np.radians(df["delta"])

    df["x_coord"] = (
        np.cos(df["delta_rad"])
        * np.cos(df["alpha_rad"])
    )

    df["y_coord"] = (
        np.cos(df["delta_rad"])
        * np.sin(df["alpha_rad"])
    )

    df["z_coord"] = np.sin(df["delta_rad"])

    df["u_div_g"] = df["u"] / (df["g"] + 1e-6)
    df["g_div_r"] = df["g"] / (df["r"] + 1e-6)
    df["r_div_i"] = df["r"] / (df["i"] + 1e-6)
    df["i_div_z"] = df["i"] / (df["z"] + 1e-6)

    df["redshift_sq"] = df["redshift"]**2
    df["redshift_cube"] = df["redshift"]**3

In [104]:
for col in [
    "spectral_type",
    "galaxy_population"
]:

    le_col = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le_col.fit(combined)

    train[col] = le_col.transform(train[col])
    test[col] = le_col.transform(test[col])

In [105]:
TARGET = "class"

X = train.drop(
    columns=["id", TARGET]
)

X_test = test.drop(
    columns=["id"]
)

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    train[TARGET]
)

In [106]:
sample_idx = (
    train.sample(
        200000,
        random_state=42
    )
    .index
)

X_sample = X.loc[sample_idx]
y_sample = y[sample_idx]

print(X_sample.shape)

(200000, 28)


In [107]:
def objective(trial):

    params = {

        "objective":
            "multi:softprob",

        "num_class":
            3,

        "tree_method":
            "hist",

        "device":
            "cuda",

        "eval_metric":
            "mlogloss",

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                1200,
                4000
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                4,
                12
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

        "min_child_weight":
            trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0,
                5
            ),

        "reg_alpha":
            trial.suggest_float(
                "reg_alpha",
                0,
                5
            ),

        "reg_lambda":
            trial.suggest_float(
                "reg_lambda",
                0,
                10
            ),

        "random_state":
            42
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, valid_idx in skf.split(
        X_sample,
        y_sample
    ):

        X_train = X_sample.iloc[train_idx]
        X_valid = X_sample.iloc[valid_idx]

        y_train = y_sample[train_idx]
        y_valid = y_sample[valid_idx]

        model = XGBClassifier(
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_valid
        )

        score = balanced_accuracy_score(
            y_valid,
            pred
        )

        scores.append(score)

    return np.mean(scores)

In [108]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-06 16:08:10,034] A new study created in memory with name: no-name-08c62276-c3e7-4cd2-8688-72fd390cdd4a
[I 2026-06-06 16:08:49,330] Trial 0 finished with value: 0.9498758918273813 and parameters: {'n_estimators': 2693, 'max_depth': 12, 'learning_rate': 0.035340372176886334, 'subsample': 0.7460190188345956, 'colsample_bytree': 0.8431187802493854, 'min_child_weight': 2, 'gamma': 3.829760386166583, 'reg_alpha': 2.85527647028844, 'reg_lambda': 4.925421751291496}. Best is trial 0 with value: 0.9498758918273813.
[I 2026-06-06 16:09:51,227] Trial 1 finished with value: 0.9527850146353728 and parameters: {'n_estimators': 3674, 'max_depth': 9, 'learning_rate': 0.016569731783717952, 'subsample': 0.8127386112799587, 'colsample_bytree': 0.7740817433140887, 'min_child_weight': 1, 'gamma': 0.9575321526434444, 'reg_alpha': 4.8951770365103195, 'reg_lambda': 6.260525925020449}. Best is trial 1 with value: 0.9527850146353728.
[I 2026-06-06 16:10:39,920] Trial 2 finished with value: 0.950565063

In [109]:
best_params = study.best_params

print(
    "Best Score:",
    study.best_value
)

print(best_params)

Best Score: 0.9534085407993387
{'n_estimators': 1870, 'max_depth': 8, 'learning_rate': 0.016995695166858015, 'subsample': 0.9267207136140596, 'colsample_bytree': 0.7563485071516991, 'min_child_weight': 9, 'gamma': 0.31599136747562945, 'reg_alpha': 0.007413522873670253, 'reg_lambda': 6.868759560345417}


In [110]:
with open(
    "best_params.json",
    "w"
) as f:

    json.dump(
        best_params,
        f,
        indent=4
    )

In [111]:
trials_df = study.trials_dataframe()

trials_df.to_csv(
    "optuna_trials.csv",
    index=False
)

In [112]:
final_params = best_params.copy()

final_params.update({

    "objective":
        "multi:softprob",

    "num_class":
        3,

    "tree_method":
        "hist",

    "device":
        "cuda",

    "eval_metric":
        "mlogloss",

    "random_state":
        42
})

model = XGBClassifier(
    **final_params
)

model.fit(
    X,
    y
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7563485071516991, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=0.31599136747562945, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.016995695166858015,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=9, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1870, n_jobs=None, num_class=3, ...)

In [113]:
test_probs = model.predict_proba(
    X_test
)

preds = np.argmax(
    test_probs,
    axis=1
)

In [114]:
submission = pd.DataFrame({

    "id":
        test["id"],

    "class":
        target_encoder.inverse_transform(
            preds
        )
})

submission.to_csv(
    "submission_xgb_optuna.csv",
    index=False
)

submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [115]:
prob_df = pd.DataFrame(
    test_probs,
    columns=target_encoder.classes_
)

prob_df.insert(
    0,
    "id",
    test["id"]
)

prob_df.to_csv(
    "xgb_optuna_prob.csv",
    index=False
)

In [116]:
from google.colab import files

files.download(
    "submission_xgb_optuna.csv"
)

files.download(
    "xgb_optuna_prob.csv"
)

files.download(
    "best_params.json"
)

files.download(
    "optuna_trials.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>